In [ ]:
import pandas as pd
from datasets import Dataset

# Sample dataset with human feedback
data = {
    'input': ["Text example 1", "Text example 2", "Text example 3"],
    'feedback': [1, 0, 1]  # 1: good response, 0: bad response
}

# Convert to Hugging Face Dataset format
df = pd.DataFrame(data)
dataset = Dataset.from_pandas(df)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Load the tokenizer and model
model_name = "Salesforce/phi-3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
reward_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples['input'], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Training arguments for the reward model
training_args = TrainingArguments(
    output_dir="./reward_model",
    evaluation_strategy="epoch",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_steps=10_000,
    save_total_limit=2,
)

# Define Trainer for the reward model
trainer = Trainer(
    model=reward_model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Train the reward model
trainer.train()


In [ ]:
import torch
from transformers import AutoModelForCausalLM, PPOTrainer, PPOConfig, PPOArguments

# Load the pre-trained phi-3 model for language generation
language_model = AutoModelForCausalLM.from_pretrained(model_name)

# Define a function to generate responses and calculate rewards
def generate_responses(inputs, model, tokenizer):
    model.eval()
    responses = []
    with torch.no_grad():
        for input_text in inputs:
            input_ids = tokenizer.encode(input_text, return_tensors='pt')
            response_ids = model.generate(input_ids, max_length=512)
            response_text = tokenizer.decode(response_ids[0], skip_special_tokens=True)
            responses.append(response_text)
    return responses

# Define a function to compute rewards
def compute_rewards(responses, reward_model, tokenizer):
    rewards = []
    for response in responses:
        inputs = tokenizer.encode(response, return_tensors='pt')
        outputs = reward_model(inputs)
        reward = outputs.logits[0][1]  # Assuming index 1 is the positive class
        rewards.append(reward.item())
    return rewards

# Generate responses for the dataset
responses = generate_responses(dataset['input'], language_model, tokenizer)

# Compute rewards for the generated responses
rewards = compute_rewards(responses, reward_model, tokenizer)

# Prepare the dataset for PPO
ppo_dataset = [{'input': input_text, 'response': response, 'reward': reward}
               for input_text, response, reward in zip(dataset['input'], responses, rewards)]

# Define PPO configuration
ppo_config = PPOConfig(
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3
)

# Define PPO arguments
ppo_args = PPOArguments(
    output_dir="./ppo_language_model",
    evaluation_strategy="epoch",
    save_steps=10_000,
    save_total_limit=2,
)

# Define PPO Trainer
ppo_trainer = PPOTrainer(
    model=language_model,
    args=ppo_args,
    train_dataset=ppo_dataset,
    reward_function=compute_rewards  # Custom reward function for PPO
)

# Fine-tune the language model with PPO
ppo_trainer.train()
